In [ ]:
# ---------------- Imports ----------------
import os
import json
import random
import yaml
from collections import defaultdict
import csv
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd
from scipy.stats import binomtest



In [ ]:
# ---------------- Args ----------------
RESULTS_FILE = "019c408d-bdfe-7734-bddb-2f2a8d0ab953-processed"
LOW_FAMILIARITY_CUTFOFF_INCLUSIVE = 5



In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]

RESULTS_PATH = os.path.join(PROJ_STORE, "evaluation", "human-evaluation", "results", "processed", f"{RESULTS_FILE}.csv")




### Prelim Analysis

---

In [ ]:
# Load data
df = pd.read_csv(RESULTS_PATH)

display(df.shape)
display(df.head(2))



In [ ]:
# Binary correctness
df["is_correct"] = (df["accurate_response"] == df["true_label"]).astype(int)

# Binary false-claim indicator
df["is_false_claim"] = (df["true_label"] == "REFUTES").astype(int)

# Binary framed indicator
#df["is_framed"] = (df["framing_type"] != "original").astype(int)

# Optional: collapse familiarity into low / high (median split)
df["low_familiarity"] = (df["familiarity_response"] <= LOW_FAMILIARITY_CUTFOFF_INCLUSIVE).astype(int)


display(df.head(2))



In [ ]:
df.describe()

In [ ]:
df["true_label"].value_counts().sort_index()

In [ ]:
df["accurate_response"].value_counts().sort_index()

In [ ]:
df["confidence_response"].value_counts().sort_index()

In [ ]:
df["familiarity_response"].value_counts().sort_index()

In [ ]:
# Refuted claims that got supported by framing_type

false_supported = df[
    (df["true_label"] == "REFUTES") &
    (df["accurate_response"] == "SUPPORTS")
]

false_supported["framing_type"].value_counts()




In [ ]:
# Refuted claims that got supported by framing_type

true_rejected = df[
    (df["true_label"] == "SUPPORTS") &
    (df["accurate_response"] == "REFUTES")
]

true_rejected["framing_type"].value_counts()




### $\chi^2$ Tests

---

In [ ]:
# All REFUTES

# Contingency table: framing × outcome
fa_table = pd.crosstab(
    df[df.true_label == "REFUTES"]["framing_type"],
    df[df.true_label == "REFUTES"]["accurate_response"]
)

fa_table



In [ ]:
chi2, p, dof, expected = chi2_contingency(fa_table)

chi2, p



In [ ]:
# All SUPPORTS
fr_table = pd.crosstab(
    df[df.true_label == "SUPPORTS"]["framing_type"],
    df[df.true_label == "SUPPORTS"]["accurate_response"]
)

fr_table



In [ ]:
chi2_fr, p_fr, _, _ = chi2_contingency(fr_table)

chi2_fr, p_fr



## Other

In [ ]:
## Modified aggregation code


# Prelim analysis
FAMILIARITY_LEVEL = [1,2,3,4,5]  # explicitly included levels


df_filt = df[df["familiarity_response"].isin(FAMILIARITY_LEVEL)].copy()

summary_table = (
    df_filt
    .groupby("framing_type")
    .agg(
        # Overall counts
        n=("is_correct", "size"),

        # Overall averages
        avg_accuracy_pct=("is_correct", "mean"),
        avg_confidence=("confidence_response", "mean"),
        avg_familiarity=("familiarity_response", "mean"),

        # Ns by truth value
        n_refutes=("true_label", lambda x: (x == "REFUTES").sum()),
        n_supports=("true_label", lambda x: (x == "SUPPORTS").sum()),

        # Accuracy by truth value
        acc_refutes=("is_correct", lambda x: x[df_filt.loc[x.index, "true_label"] == "REFUTES"].mean()),
        acc_supports=("is_correct", lambda x: x[df_filt.loc[x.index, "true_label"] == "SUPPORTS"].mean()),

        # --- Core framing-effect metrics ---
        # False acceptance: P(SUPPORTS | REFUTES)
        false_accept_rate=(
            "accurate_response",
            lambda x: (x[df_filt.loc[x.index, "true_label"] == "REFUTES"] == "SUPPORTS").mean()
        ),

        # False rejection: P(REFUTES | SUPPORTS)
        false_reject_rate=(
            "accurate_response",
            lambda x: (x[df_filt.loc[x.index, "true_label"] == "SUPPORTS"] == "REFUTES").mean()
        ),

        # Confidence by truth value
        conf_refutes=("confidence_response", lambda x: x[df_filt.loc[x.index, "true_label"] == "REFUTES"].mean()),
        conf_supports=("confidence_response", lambda x: x[df_filt.loc[x.index, "true_label"] == "SUPPORTS"].mean()),

        # Familiarity by truth value
        fam_refutes=("familiarity_response", lambda x: x[df_filt.loc[x.index, "true_label"] == "REFUTES"].mean()),
        fam_supports=("familiarity_response", lambda x: x[df_filt.loc[x.index, "true_label"] == "SUPPORTS"].mean()),
    )
    .reset_index()
)

# ---------------- Formatting ----------------

# Percentages
summary_table["avg_accuracy_pct"] = (summary_table["avg_accuracy_pct"] * 100).round(2)
summary_table["acc_refutes"] = (summary_table["acc_refutes"] * 100).round(2)
summary_table["acc_supports"] = (summary_table["acc_supports"] * 100).round(2)
summary_table["false_accept_rate"] = (summary_table["false_accept_rate"] * 100).round(2)
summary_table["false_reject_rate"] = (summary_table["false_reject_rate"] * 100).round(2)

# Scalars
for col in [
    "avg_confidence", "conf_refutes", "conf_supports",
    "avg_familiarity", "fam_refutes", "fam_supports"
]:
    summary_table[col] = summary_table[col].round(2)

# ---------------- Ordering ----------------

summary_table["framing_type"] = pd.Categorical(
    summary_table["framing_type"],
    categories=["original"] + [
        f for f in summary_table["framing_type"].unique()
        if f != "original"
    ],
    ordered=True
)

summary_table = summary_table.sort_values("framing_type").reset_index(drop=True)

# ---------------- Overall row ----------------

overall_row = pd.DataFrame([{
    "framing_type": "Overall",
    "n": df_filt.shape[0],

    "avg_accuracy_pct": round(df_filt["is_correct"].mean() * 100, 2),
    "avg_confidence": round(df_filt["confidence_response"].mean(), 2),
    "avg_familiarity": round(df_filt["familiarity_response"].mean(), 2),

    "n_refutes": (df_filt["true_label"] == "REFUTES").sum(),
    "n_supports": (df_filt["true_label"] == "SUPPORTS").sum(),

    "acc_refutes": round(df_filt[df_filt.true_label == "REFUTES"]["is_correct"].mean() * 100, 2),
    "acc_supports": round(df_filt[df_filt.true_label == "SUPPORTS"]["is_correct"].mean() * 100, 2),

    "false_accept_rate": round(
        (df_filt[(df_filt.true_label == "REFUTES") & (df_filt.accurate_response == "SUPPORTS")].shape[0]
         / max(1, (df_filt.true_label == "REFUTES").sum())) * 100,
        2
    ),

    "false_reject_rate": round(
        (df_filt[(df_filt.true_label == "SUPPORTS") & (df_filt.accurate_response == "REFUTES")].shape[0]
         / max(1, (df_filt.true_label == "SUPPORTS").sum())) * 100,
        2
    ),

    "conf_refutes": round(df_filt[df_filt.true_label == "REFUTES"]["confidence_response"].mean(), 2),
    "conf_supports": round(df_filt[df_filt.true_label == "SUPPORTS"]["confidence_response"].mean(), 2),

    "fam_refutes": round(df_filt[df_filt.true_label == "REFUTES"]["familiarity_response"].mean(), 2),
    "fam_supports": round(df_filt[df_filt.true_label == "SUPPORTS"]["familiarity_response"].mean(), 2),
}])

final_table = pd.concat([summary_table, overall_row], ignore_index=True)

display(final_table)



In [ ]:


# ---------------- Familiarity control ----------------
# Choose exactly which familiarity levels to include
FAMILIARITY_LEVELS = [4,5]   # e.g., [1], [3], [4,5], [1,2,3]

# ---------------- Restrict to selected-familiarity false claims ----------------
lf_false = df[
    (df["true_label"] == "REFUTES") &
    (df["familiarity_response"].isin(FAMILIARITY_LEVELS))
].copy()

# ---------------- Define false acceptance ----------------
lf_false["fooled"] = (lf_false["accurate_response"] == "SUPPORTS").astype(int)

# ---------------- Summary table ----------------
summary_table = (
    lf_false
    .groupby("framing_type")
    .agg(
        false_acceptance_rate=("fooled", "mean"),
        avg_confidence=("confidence_response", "mean"),
        n=("fooled", "size"),
        n_fooled=("fooled", "sum")
    )
    .reset_index()
)

# ---------------- One-sample binomial test vs 0.5 ----------------
summary_table["p_value_vs_0_5"] = summary_table.apply(
    lambda row: binomtest(
        k=int(row["n_fooled"]),
        n=int(row["n"]),
        p=0.5,
        alternative="two-sided"
    ).pvalue,
    axis=1
)

# ---------------- Sort for presentation ----------------
summary_table = summary_table.sort_values(
    "false_acceptance_rate",
    ascending=False
).reset_index(drop=True)

display(summary_table)